# 02 - OCR y Extracción de Datos con EasyOCR

Este notebook lee los recortes generados por el notebook 01 y extrae automáticamente los valores numéricos usando EasyOCR.

**Entrada**: `data/output/crops_index_general.csv`  
**Salida**: `data/output/ocr_results.csv` con columnas: `form_id`, `field`, `ocr_value`

## 1. Configuración y imports

In [1]:
from pathlib import Path
import pandas as pd
import sys
import cv2
import numpy as np
import easyocr
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Configuración de paths
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

DATA_DIR = ROOT / 'data'
OUTPUT_DIR = DATA_DIR / 'output'
CROPS_INDEX_FILE = OUTPUT_DIR / 'crops_index_general.csv'
OCR_RESULTS_FILE = OUTPUT_DIR / 'ocr_results.csv'

print(f'✓ Rutas configuradas')
print(f'  - Índice de recortes: {CROPS_INDEX_FILE}')
print(f'  - Archivo de resultados: {OCR_RESULTS_FILE}')

✓ Rutas configuradas
  - Índice de recortes: c:\Users\default.LAPTOP-M81T5L1M\Desktop\2026-1\CIBERSEGURIDAD\deteccion-fraude-d14\data\output\crops_index_general.csv
  - Archivo de resultados: c:\Users\default.LAPTOP-M81T5L1M\Desktop\2026-1\CIBERSEGURIDAD\deteccion-fraude-d14\data\output\ocr_results.csv


## 2. Cargar índice de recortes

In [2]:
# Verificar que el archivo de índice existe
if not CROPS_INDEX_FILE.exists():
    raise FileNotFoundError(f' No se encontró {CROPS_INDEX_FILE}. Ejecuta el notebook 01 primero.')

# Cargar CSV de índice
crops_df = pd.read_csv(CROPS_INDEX_FILE)

print(f'✓ Índice cargado: {len(crops_df)} registros')
print(f'\nPrimeras filas:')
print(crops_df.head(9))
print(f'\nCampos únicos: {crops_df["field"].nunique()}')
print(crops_df['field'].unique())

✓ Índice cargado: 153 registros

Primeras filas:
   form_id              field  \
0  E14_001  total_sufragantes   
1  E14_001      votos_en_urna   
2  E14_001  votos_incinerados   
3  E14_001  votos_candidato_1   
4  E14_001  votos_candidato_2   
5  E14_001       votos_blanco   
6  E14_001        votos_nulos   
7  E14_001  votos_no_marcados   
8  E14_001         total_mesa   

                                           crop_path   x1   y1   x2    y2  \
0  C:/Users/muril/Documents/GitHub/deteccion-frau...   48  417  304   463   
1  C:/Users/muril/Documents/GitHub/deteccion-frau...  357  417  611   463   
2  C:/Users/muril/Documents/GitHub/deteccion-frau...  672  422  927   462   
3  C:/Users/muril/Documents/GitHub/deteccion-frau...  674  552  936   617   
4  C:/Users/muril/Documents/GitHub/deteccion-frau...  670  710  930   782   
5  C:/Users/muril/Documents/GitHub/deteccion-frau...  669  825  925   876   
6  C:/Users/muril/Documents/GitHub/deteccion-frau...  670  875  925   923   
7  C

## 3. Inicializar OCR Reader

In [3]:
print('Inicializando EasyOCR...')
# Usar inglés para números y caracteres universales
reader = easyocr.Reader(['en', 'es'], gpu=False, verbose=False)
print(' OCR Reader listo')

Inicializando EasyOCR...


✓ OCR Reader listo


## 4. Función de procesamiento de imagen y OCR

In [6]:
def preprocess_image_for_ocr(image_path, target_size=(300, 100)):
   
    try:
        # Verificar que el archivo existe
        if not Path(image_path).exists():
            return None
            
        # Cargar imagen
        img = cv2.imread(str(image_path))
        
        if img is None:
            return None
        
        # Convertir a escala de grises
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Aplicar CLAHE (Contrast Limited Adaptive Histogram Equalization)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)
        
        # Aplicar umbralización adaptativa
        binary = cv2.adaptiveThreshold(
            enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 11, 2
        )
        
        # Redimensionar
        resized = cv2.resize(binary, target_size)
        
        return resized
    except Exception as e:
        print(f'Error procesando {image_path}: {e}')
        return None


def extract_text_with_ocr(image_path, reader):
   
    try:
        # Verificar que existe el archivo
        if not Path(image_path).exists():
            return '', 0
        
        # Procesar imagen
        processed_img = preprocess_image_for_ocr(image_path)
        
        if processed_img is None:
            return '', 0
        
        # Ejecutar OCR
        results = reader.readtext(processed_img, detail=1)
        
        if not results:
            return '', 0
        
        # Extraer texto y confianza
        texts = []
        confidences = []
        
        for (bbox, text, confidence) in results:
            texts.append(text.strip())
            confidences.append(confidence)
        
        # Concatenar todos los textos
        full_text = ''.join(texts)
        
        # Calcular confianza promedio
        avg_confidence = sum(confidences) / len(confidences) if confidences else 0
        
        return full_text, avg_confidence
        
    except Exception as e:
        return '', 0


def clean_ocr_value(value_str):
 
    if not value_str:
        return ''
    
    # Mantener solo dígitos
    digits = ''.join(c for c in value_str if c.isdigit())
    
    return digits

print(' Funciones de OCR definidas')

 Funciones de OCR definidas


## 5. Procesar todos los recortes con OCR

In [7]:
print('Extrayendo valores con OCR...')
print(f'Total de recortes a procesar: {len(crops_df)}')

# Preparar lista para resultados
ocr_results = []

# Procesar cada recorte
for idx, row in tqdm(crops_df.iterrows(), total=len(crops_df), desc='OCR'):
    form_id = row['form_id']
    field = row['field']
    crop_path = row['crop_path']
    
    # Ejecutar OCR
    raw_text, confidence = extract_text_with_ocr(crop_path, reader)
    
    # Limpiar resultado
    ocr_value = clean_ocr_value(raw_text)
    
    # Guardar resultado
    ocr_results.append({
        'form_id': form_id,
        'field': field,
        'ocr_value': ocr_value,
        'raw_text': raw_text,
        'confidence': confidence
    })

# Crear DataFrame
results_df = pd.DataFrame(ocr_results)

print(f'\n✓ OCR completado: {len(results_df)} registros extraídos')
print(f'\nPrimeros 15 resultados:')
print(results_df[['form_id', 'field', 'ocr_value', 'confidence']].head(15))

Extrayendo valores con OCR...
Total de recortes a procesar: 153


OCR: 100%|██████████| 153/153 [00:00<00:00, 6409.53it/s]


✓ OCR completado: 153 registros extraídos

Primeros 15 resultados:
    form_id              field ocr_value  confidence
0   E14_001  total_sufragantes                     0
1   E14_001      votos_en_urna                     0
2   E14_001  votos_incinerados                     0
3   E14_001  votos_candidato_1                     0
4   E14_001  votos_candidato_2                     0
5   E14_001       votos_blanco                     0
6   E14_001        votos_nulos                     0
7   E14_001  votos_no_marcados                     0
8   E14_001         total_mesa                     0
9   E14_002  total_sufragantes                     0
10  E14_002      votos_en_urna                     0
11  E14_002  votos_incinerados                     0
12  E14_002  votos_candidato_1                     0
13  E14_002  votos_candidato_2                     0
14  E14_002       votos_blanco                     0


## 6. Análisis de calidad de OCR

In [9]:
# Estadísticas de confianza
print('ESTADÍSTICAS DE OCR')
print('=' * 60)

print(f'\nConfianza promedio: {results_df["confidence"].mean():.4f}')
print(f'Confianza mínima: {results_df["confidence"].min():.4f}')
print(f'Confianza máxima: {results_df["confidence"].max():.4f}')

# Campos vacíos
empty_fields = results_df[results_df['ocr_value'].isna() | (results_df['ocr_value'] == '')]
print(f'\nCampos vacíos (no detectados): {len(empty_fields)}')

if len(empty_fields) > 0:
    print('\nDetalle de campos vacíos:')
    empty_summary = empty_fields.groupby(['form_id', 'field']).size().reset_index(name='count')
    print(empty_summary.head(10))

# Campos de baja confianza
low_confidence = results_df[results_df['confidence'] < 0.5]
print(f'\nCampos con confianza < 0.5: {len(low_confidence)}')

if len(low_confidence) > 0:
    print('\nDetalle de baja confianza (primeros 10):')
    print(low_confidence[['form_id', 'field', 'ocr_value', 'confidence']].head(10))

# Distribución de confianza
print(f'\nDistribución de confianza:')
bins = [0, 0.3, 0.6, 0.8, 1.0]
labels = ['0-0.3', '0.3-0.6', '0.6-0.8', '0.8-1.0']
results_df['confidence_bin'] = pd.cut(results_df['confidence'], bins=bins, labels=labels)
print(results_df['confidence_bin'].value_counts().sort_index())

ESTADÍSTICAS DE OCR

Confianza promedio: 0.0000
Confianza mínima: 0.0000
Confianza máxima: 0.0000

Campos vacíos (no detectados): 153

Detalle de campos vacíos:
   form_id              field  count
0  E14_001         total_mesa      1
1  E14_001  total_sufragantes      1
2  E14_001       votos_blanco      1
3  E14_001  votos_candidato_1      1
4  E14_001  votos_candidato_2      1
5  E14_001      votos_en_urna      1
6  E14_001  votos_incinerados      1
7  E14_001  votos_no_marcados      1
8  E14_001        votos_nulos      1
9  E14_002         total_mesa      1

Campos con confianza < 0.5: 153

Detalle de baja confianza (primeros 10):
   form_id              field ocr_value  confidence
0  E14_001  total_sufragantes                     0
1  E14_001      votos_en_urna                     0
2  E14_001  votos_incinerados                     0
3  E14_001  votos_candidato_1                     0
4  E14_001  votos_candidato_2                     0
5  E14_001       votos_blanco                

## 7. Guardar resultados en CSV

In [10]:
# Crear DataFrame final con SOLO las 3 columnas requeridas
final_results = results_df[['form_id', 'field', 'ocr_value']].copy()

# Guardar CSV
final_results.to_csv(OCR_RESULTS_FILE, index=False, encoding='utf-8')

print(f'✓ Archivo guardado: {OCR_RESULTS_FILE}')
print(f'\nÚltimas 15 filas del archivo generado:')
print(final_results.tail(15))

# Verificar integridad
verify_df = pd.read_csv(OCR_RESULTS_FILE)
print(f'\n✓ Verificación: {len(verify_df)} registros guardados')
print(f'Columnas: {list(verify_df.columns)}')

✓ Archivo guardado: c:\Users\default.LAPTOP-M81T5L1M\Desktop\2026-1\CIBERSEGURIDAD\deteccion-fraude-d14\data\output\ocr_results.csv

Últimas 15 filas del archivo generado:
     form_id              field ocr_value
138  E14_016  votos_candidato_1          
139  E14_016  votos_candidato_2          
140  E14_016       votos_blanco          
141  E14_016        votos_nulos          
142  E14_016  votos_no_marcados          
143  E14_016         total_mesa          
144  E14_017  total_sufragantes          
145  E14_017      votos_en_urna          
146  E14_017  votos_incinerados          
147  E14_017  votos_candidato_1          
148  E14_017  votos_candidato_2          
149  E14_017       votos_blanco          
150  E14_017        votos_nulos          
151  E14_017  votos_no_marcados          
152  E14_017         total_mesa          

✓ Verificación: 153 registros guardados
Columnas: ['form_id', 'field', 'ocr_value']


## 8. Resumen de resultados por formulario

In [11]:
# Resumen por formulario
print('\n RESUMEN POR FORMULARIO')
print('=' * 80)

for form_id in sorted(results_df['form_id'].unique()):
    form_data = results_df[results_df['form_id'] == form_id]
    total_fields = len(form_data)
    empty_count = len(form_data[form_data['ocr_value'] == ''])
    avg_conf = form_data['confidence'].mean()
    
    print(f'\n{form_id}:')
    print(f'  • Total campos: {total_fields}')
    print(f'  • Campos extraídos: {total_fields - empty_count}/{total_fields}')
    print(f'  • Confianza promedio: {avg_conf:.4f}')
    
    # Mostrar valores extraídos
    print(f'  • Valores:')
    for _, row in form_data.iterrows():
        value_display = row['ocr_value'] if row['ocr_value'] else '[VACÍO]'
        print(f'    - {row["field"]}: {value_display}')


 RESUMEN POR FORMULARIO

E14_001:
  • Total campos: 9
  • Campos extraídos: 0/9
  • Confianza promedio: 0.0000
  • Valores:
    - total_sufragantes: [VACÍO]
    - votos_en_urna: [VACÍO]
    - votos_incinerados: [VACÍO]
    - votos_candidato_1: [VACÍO]
    - votos_candidato_2: [VACÍO]
    - votos_blanco: [VACÍO]
    - votos_nulos: [VACÍO]
    - votos_no_marcados: [VACÍO]
    - total_mesa: [VACÍO]

E14_002:
  • Total campos: 9
  • Campos extraídos: 0/9
  • Confianza promedio: 0.0000
  • Valores:
    - total_sufragantes: [VACÍO]
    - votos_en_urna: [VACÍO]
    - votos_incinerados: [VACÍO]
    - votos_candidato_1: [VACÍO]
    - votos_candidato_2: [VACÍO]
    - votos_blanco: [VACÍO]
    - votos_nulos: [VACÍO]
    - votos_no_marcados: [VACÍO]
    - total_mesa: [VACÍO]

E14_003:
  • Total campos: 9
  • Campos extraídos: 0/9
  • Confianza promedio: 0.0000
  • Valores:
    - total_sufragantes: [VACÍO]
    - votos_en_urna: [VACÍO]
    - votos_incinerados: [VACÍO]
    - votos_candidato_1: [VACÍ

## 9. Alertas y validaciones

In [12]:
print('\n  ALERTAS Y VALIDACIONES')
print('=' * 80)

alerts = []

# Alerta 1: Campos completamente vacíos
totally_empty = results_df[results_df['ocr_value'] == '']
if len(totally_empty) > 0:
    alerts.append(f'  {len(totally_empty)} campos NO LEGIBLES por OCR')
    print(f'\n {len(totally_empty)} CAMPOS NO LEGIBLES')
    empty_by_field = totally_empty.groupby('field').size().reset_index(name='count')
    print(empty_by_field.to_string(index=False))

# Alerta 2: Baja confianza
low_conf = results_df[results_df['confidence'] < 0.4]
if len(low_conf) > 0:
    alerts.append(f'  {len(low_conf)} campos con confianza < 0.4')
    print(f'\n {len(low_conf)} CAMPOS CON BAJA CONFIANZA (< 0.4)')
    low_conf_summary = low_conf[['form_id', 'field', 'ocr_value', 'confidence']].head(10)
    print(low_conf_summary.to_string(index=False))

# Alerta 3: Valores no numéricos
results_df['is_numeric'] = results_df['ocr_value'].str.match(r'^\d+$')
non_numeric = results_df[~results_df['is_numeric'] & (results_df['ocr_value'] != '')]
if len(non_numeric) > 0:
    alerts.append(f'ℹ  {len(non_numeric)} campos con caracteres no numéricos')
    print(f'\n️  {len(non_numeric)} VALORES CON CARACTERES NO NUMÉRICOS')
    non_numeric_sample = non_numeric[['form_id', 'field', 'ocr_value']].head(10)
    print(non_numeric_sample.to_string(index=False))

if not alerts:
    print('\n✓ No se encontraron alertas significativas')

print(f'\n\n PROCESO COMPLETADO')
print(f'Archivo listo para análisis de anomalías: {OCR_RESULTS_FILE}')


  ALERTAS Y VALIDACIONES

 153 CAMPOS NO LEGIBLES
            field  count
       total_mesa     17
total_sufragantes     17
     votos_blanco     17
votos_candidato_1     17
votos_candidato_2     17
    votos_en_urna     17
votos_incinerados     17
votos_no_marcados     17
      votos_nulos     17

 153 CAMPOS CON BAJA CONFIANZA (< 0.4)
form_id             field ocr_value  confidence
E14_001 total_sufragantes                     0
E14_001     votos_en_urna                     0
E14_001 votos_incinerados                     0
E14_001 votos_candidato_1                     0
E14_001 votos_candidato_2                     0
E14_001      votos_blanco                     0
E14_001       votos_nulos                     0
E14_001 votos_no_marcados                     0
E14_001        total_mesa                     0
E14_002 total_sufragantes                     0


 PROCESO COMPLETADO
Archivo listo para análisis de anomalías: c:\Users\default.LAPTOP-M81T5L1M\Desktop\2026-1\CIBERSEGURIDAD\dete